### Prefix Tuning Test Run

#### What is Prefix Tuning?
- Way to guide/influence the output of LLMs withut having to fine-tune all of the model parameters (ie. ChatGPT3 has 175B parameters)
- **Definition**: Essentially is a hint that guides an LLM to generate text in different styles without modifiying most parameters - therefore requires less data.
    - Small module or set of trainable parameters (the prefix). 
- **Prefix general definition** --> extra info added at beginning of initial input to guide model response. Set of specific words or phrases added to beginning of user prompt which acts as signal to guide model to desired style/tone/task. Is a pre-set for model behavior. 
- **Prefix in prefix-tuning** --> set of special learned tokens(hidden numbers) that act as instructions for the model, thus influencing how the LLM generates text. 
    - Utilizes **prefix embedding** a vector of numbers
    - Prefixes can be *swapped* to change styles easily
    - The prefix produces a sequence of embeddings that are prepended to the input sequence before being fed into the model --> *it is not a sentence* but rather a learned representation of the behavior we want.
    - ex.) If model's hidden size is 768 dimensions, then each prefix token is a 768 vector --> we train on these vectors.

#### Training Process
- Load pre-trained model --> LLMs already have a strong understanding of langauge, grammar and semantics that we can build off!
- Intialize random set of prefix embeddings (tokens that we want to learn)
- Prepend to the model input **at each layer** of the transformer
    - ie.) We give a little nudge/hint in the direction we want *at every stage* of model's processing.
- During training, only prefix embeddings are updated - rest of the model remains **frozen**.
- How it's trained:
    - Minimizing error on a task-specific dataset. ie.) If we want sentiment analysis --> use a sentiment analyiss dataset  
- As the prefixes get better, model outputs resemble our desired style/behavior!

#### Training Process, In Depth, The Supervised Case
- Pass Input-Output Pairs.)
    - ex.) Neutral Sentence (Input). Formal sentence (Target/label).
- Tokenize the sentences --> conver to numbers we can process
- Forward pass --> input embeddings, expand prefix embeddings, concatnate prefix with input, transformer layers process the combined input
- Loss Calculation - model makes some output sequence. Compare output to target/label using loss function like **Cross-Entropy Loss**
    - Cross Entropy Loss - measures how close predicted tokens are to actual target tokens.
- Backprogation - **only update** the **prefix embeddings**
- Repeat - Train over many epochs (complete passes through dataset)
- Inference - try on new text 
#### Important Vocabulary
- **Hidden Size** - The hidden size of a model refers to the dimensionality/size of the vectors that represent information at each layer of the model --> size of the feature space where model processes + stores info.
    - NNs have layers that data is passed through --> each layer processes data and transforms it into a more abstract/higher-level representation ex.) lines and edges --> basic shapes --> facial features 
    - LLM have special layers called transformers.
    - **Extra - what changes across layers?**
        - Attention Mechanisms - each layer uses **attention** to focus/pay attention to different parts of the sentence, using that toa djust the token/embedding represtations based on **context**.
        - Feedforward - these layers apply non-linear + complex transformations, give the words/token representations with more complex patterns.
        - Residual Connections + Layer Normalization - ensure stable training and allow deeper layers to build on knowledge of earliers. 
    - Hidden Size determines how much info each layer can hold/process at any given moment.
- **Hidden Size as a Vector** - For each token (split off word), they each have their own vector of numbers associated with them. If hidden size = 768 (like with GPT2) then each token has 768 numbers that capture the meaning and context of the word --> these numbers are adjusted as they pass through each layer as the model learns its meaning better.
- **Token size of prefix** - Simply how many learnable vectors/prefix embeddings you add to the input before getting processed. Each is a guide/hint. Prefix of 5 tokens = 5 learnable vectors

#### Code Toy Example 

In [ ]:
import torch
from torch import nn, optim
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from torch.utils.data import DataLoader, Dataset

# Step 1: Define a simple dataset of neutral sentences and their formal equivalents
class StyleTransferDataset(Dataset):
    def __init__(self, neutral_sentences, formal_sentences, tokenizer, max_length=50):
        self.neutral_sentences = neutral_sentences
        self.formal_sentences = formal_sentences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.neutral_sentences)

    def __getitem__(self, idx):
        # Tokenize the neutral (input) sentence
        input_encoding = self.tokenizer(
            self.neutral_sentences[idx],
            return_tensors='pt',
            max_length=self.max_length,
            truncation=True,
            padding='max_length'
        )

        # Tokenize the formal (target) sentence
        target_encoding = self.tokenizer(
            self.formal_sentences[idx],
            return_tensors='pt',
            max_length=self.max_length,
            truncation=True,
            padding='max_length'
        )

        return {
            'input_ids': input_encoding['input_ids'].squeeze(),
            'attention_mask': input_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze()
        }

# Step 2: Initialize the tokenizer and dataset
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Example sentences for the dummy run
neutral_sentences = [
    "I can't attend the meeting.",
    "She won't be here tomorrow.",
    "He is late for the event.",
    "I'm busy right now.",
    "They don't like the proposal."
]

formal_sentences = [
    "I regret to inform you that I am unable to attend the meeting.",
    "She will not be present tomorrow.",
    "He has arrived late for the event.",
    "I am currently occupied.",
    "They are not in favor of the proposal."
]

# Create the dataset and data loader
dataset = StyleTransferDataset(neutral_sentences, formal_sentences, tokenizer)
data_loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Step 3: Load the pre-trained GPT-2 model and freeze its parameters
model = GPT2LMHeadModel.from_pretrained('gpt2')
for param in model.parameters():
    param.requires_grad = False  # Freeze the entire model

# Step 4: Create the prefix embedding (learnable parameters)
class PrefixTuning(nn.Module):
    def __init__(self, model, prefix_length=5):
        super(PrefixTuning, self).__init__()
        self.model = model
        self.prefix_length = prefix_length
        self.hidden_size = model.config.hidden_size  # GPT-2 hidden size

        # Initialize a learnable prefix tensor
        self.prefix = nn.Parameter(torch.randn(prefix_length, self.hidden_size))

    def forward(self, input_ids, attention_mask=None, labels=None):
        # Get input embeddings from GPT-2 tokenizer
        input_embeddings = self.model.transformer.wte(input_ids)

        # Expand prefix to match batch size
        batch_size = input_ids.shape[0]
        expanded_prefix = self.prefix.unsqueeze(0).expand(batch_size, -1, -1)

        # Concatenate prefix embeddings with input embeddings
        combined_embeddings = torch.cat((expanded_prefix, input_embeddings), dim=1)

        # Adjust attention mask to account for prefix
        if attention_mask is not None:
            prefix_attention = torch.ones((batch_size, self.prefix_length), dtype=attention_mask.dtype).to(attention_mask.device)
            attention_mask = torch.cat((prefix_attention, attention_mask), dim=1)

        # Pass the combined embeddings through GPT-2
        outputs = self.model(inputs_embeds=combined_embeddings, attention_mask=attention_mask, labels=labels)
        return outputs

# Initialize the prefix-tuning model
prefix_tuning_model = PrefixTuning(model)

# Step 5: Set up the optimizer and loss function
optimizer = optim.Adam(prefix_tuning_model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# Step 6: Training loop
num_epochs = 5
prefix_tuning_model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for batch in data_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['labels']

        outputs = prefix_tuning_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(data_loader):.4f}")

# Step 7: Inference - Test the model on new sentences
prefix_tuning_model.eval()

new_sentences = [
    "I don't like this idea.",
    "We can't meet today.",
    "She is busy now."
]

for sentence in new_sentences:
    input_ids = tokenizer(sentence, return_tensors='pt')['input_ids']
    attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = prefix_tuning_model(input_ids=input_ids, attention_mask=attention_mask)
        generated_ids = outputs.logits.argmax(-1)

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    print(f"Original: {sentence}")
    print(f"Formal: {generated_text}\n")
